# 02. Viewing Downloaded Manuscript Images

This notebook lets you browse manuscript facsimile images downloaded through the project's IIIF pipeline.

Images are organized in structured folders under `images/`, linked directly to the catalogue metadata in `data/csv/MANUSCRIPTS_IMAGES_INDEX.csv` and `data/csv/PAGE_LEVEL_METADATA.csv`.

## 1. Index of Downloaded Manuscripts

Let's load the central index to see all downloaded manuscripts, their shelfmarks, holding libraries, and page counts.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from PIL import Image
from IPython.display import display

# Robust path resolution (works from code/notebooks/ or project root)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "metadata").exists())
IMAGES_DIR = PROJECT_ROOT / "data" / "manuscripts"
CSV_DIR = PROJECT_ROOT / "data" / "metadata" / "raw"

# Load downloaded manuscripts index
index_file = CSV_DIR / "MANUSCRIPTS_IMAGES_INDEX.csv"
if index_file.exists():
    df_index = pd.read_csv(index_file)
    display(df_index[["library_code", "shelfmark", "title", "downloaded_pages", "folder_path"]])
else:
    print("Index file not found yet. Listing folders in data/manuscripts/:")
    for f in sorted(IMAGES_DIR.iterdir()):
        if f.is_dir() and not f.name.startswith("."):
            print(" -", f.name)

## 2. Select and Inspect a Manuscript

Choose a manuscript by setting `selected_folder` below (copy from the index table above, or browse any folder in `images/`).

You can control how many pages to display using `max_pages` (to avoid scrolling through hundreds of pages at once).

In [ ]:
# --- Configuration: Choose manuscript folder and page display limit ---
selected_folder = "990000463750205171_NLI_Ms._Heb._6694_28"  # e.g. "990001314740205171_BnF_Ms._hebr._1105", "990000846580205171_JTS_Ms._9121", etc.
max_pages_to_show = 5  # Set to None to show all pages in the folder

target_dir = IMAGES_DIR / selected_folder

if not target_dir.exists():
    print(f"Directory not found: {target_dir}")
else:
    # 1. Display metadata if metadata.json exists
    meta_path = target_dir / "metadata.json"
    if meta_path.exists():
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)
        print("=== Manuscript Metadata ===")
        print(f"Title:     {meta.get('title')}")
        print(f"Library:   {meta.get('holding_library')} ({meta.get('library_code')})")
        print(f"Shelfmark: {meta.get('shelfmark')}")
        print(f"Script:    {meta.get('script') or 'Not recorded'}")
        if meta.get("colophon"):
            print(f"Colophon:  {meta.get('colophon')[:120]}...")
        print(f"Pages:     {meta.get('downloaded_pages')} downloaded\n")
        
    # 2. Display pages
    image_paths = sorted(target_dir.glob("*.jpg"))
    display_paths = image_paths[:max_pages_to_show] if max_pages_to_show else image_paths
    
    print(f"Displaying {len(display_paths)} of {len(image_paths)} pages from: {selected_folder}\n")
    
    for idx, img_path in enumerate(display_paths, 1):
        with Image.open(img_path) as im:
            w, h = im.size
            file_kb = img_path.stat().st_size / 1024
            print(f"--- Page {idx}: {img_path.name} | Dimensions: {w}x{h} px | Size: {file_kb:.1f} KB ---")
            display(im.copy())